In [2]:
import Pkg

# Activate your MyProject environment
Pkg.activate(raw"C:\Users\HP\Code\Master thesis\Model\MyJulia19")

# Confirm the active environment
println("Active environment: ", Base.active_project())

  Activating project at `C:\Users\HP\Code\Master thesis\Model\MyJulia19`


Active environment: C:\Users\HP\Code\Master thesis\Model\MyJulia19\Project.toml


In [3]:
using Random, Plots
using Zygote, ForwardDiff
using OrdinaryDiffEq, DiffEqSensitivity
using LinearAlgebra
using Statistics
using ProgressBars, Printf
using Flux
using Flux.Optimise: update!
using Flux.Losses: mae, mse
using BSON: @save, @load
using DelimitedFiles
using YAML
using MLFlowClient

In [4]:
pwd()

"C:\\Users\\HP\\Code\\Master thesis\\Model\\Xylan\\Model Diff+Lid"

In [6]:
# length(lr_trace)
# iter

In [7]:
ENV["GKSwstype"] = "100"

"100"

In [85]:
# Create MLFlow instance
mlf = MLFlow("http://localhost:5000/api")

# Initiate new experiment

# experiment_id = createexperiment(mlf, "Xylan_lid - exps")

mlf_exp = getexperimentbyname(mlf, "Xylan_lid - exps")
experiment_id = mlf_exp.experiment_id

# Create a run in the new experiment

exprun = createrun(mlf, experiment_id)
# old_run_id = "2f81b0c9abfb49e6904f0ead991ed1d3"  # from UI
# exprun = getrun(mlf, old_run_id)


Run(
    info = RunInfo(
    run_id = "42c5491692b34e4d9abf0e39cd355364", 
    run_name = "sneaky-fox-736", 
    experiment_id = "509917977778309868", 
    status = MLFlowClient.RunStatus.RUNNING, 
    start_time = 0, 
    end_time = nothing, 
    artifact_uri = "mlflow-artifacts:/509917977778309868/42c5491692b34e4d9abf0e39cd355364/artifacts", 
    lifecycle_stage = "active"
), 
    data = RunData(
    metrics = Metric[], 
    params = Param[], 
    tags = Tag[Tag(
    key = "mlflow.runName", 
    value = "sneaky-fox-736"
)]
), 
    inputs = RunInputs(
    dataset_inputs = DatasetInput[], 
    model_inputs = ModelInput[]
), 
    outputs = MLFlowClient.RunOutputs(
    model_outputs = ModelOutput[]
)
)

In [86]:
conf = YAML.load_file("./config2.yaml")

Dict{Any, Any} with 15 entries:
  "lr_max"        => 0.0001
  "grad_max"      => 100.0
  "maxiters"      => 10000
  "n_epoch"       => 1500
  "lr_min"        => 1.0e-5
  "is_restart"    => false
  "lr_decay"      => 0.5
  "lr_decay_step" => 500
  "nc"            => 1
  "expr_name"     => "Lid-6s7r1c-03"
  "nr"            => 5
  "n_plot"        => 10
  "ns"            => 6
  "lb"            => 1.0e-8
  "w_decay"       => 1.0e-8

In [88]:
expr_name = conf["expr_name"]
fig_path = string("./results_lid/", expr_name, "/figs")
ckpt_path = string("./results_lid/", expr_name, "/checkpoint")
config_path = "./results_lid/$expr_name/config.yaml"

is_restart = Bool(conf["is_restart"])
ns = Int64(conf["ns"])
nr = Int64(conf["nr"])
nc = Int64(conf["nc"])
nss = Int64(conf["ns"]) - Int64(conf["nc"])
lb = Float64(conf["lb"])
n_epoch = Int64(conf["n_epoch"])
n_plot = Int64(conf["n_plot"])
grad_max = Float64(conf["grad_max"])
maxiters = Int64(conf["maxiters"])

lr_max = Float64(conf["lr_max"])
lr_min = Float64(conf["lr_min"])
lr_decay = Float64(conf["lr_decay"])
lr_decay_step = Int64(conf["lr_decay_step"])
w_decay = Float64(conf["w_decay"])

1.0e-8

In [91]:
llb = lb;
global p_cutoff = -1.0

const l_exp = 1:30
n_exp = length(l_exp)

l_train = []
l_val = []
for i = 1:n_exp
    j = l_exp[i]
    if !(j in [7,8,9,10,11,12,13,14,15,16,17,18])
        push!(l_train, i)
    else
        push!(l_val, i)
    end
end

opt = Flux.Optimiser(
  ExpDecay(lr_max, lr_decay, length(l_train) * lr_decay_step, lr_min),
  ADAMW(lr_max, (0.9, 0.999), w_decay),
);

#opt = Flux.Optimise.ADAM(0.00001,(0.9,0.999))

In [92]:
l_train, l_val, n_exp

(Any[1, 2, 3, 4, 5, 6, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30], Any[7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18], 30)

In [95]:
if !is_restart
    if ispath(fig_path)
        rm(fig_path, recursive = true)
    end
    if ispath(ckpt_path)
        rm(ckpt_path, recursive = true)
    end
end

if ispath("./results_lid") == false
    mkdir("./results_lid")
end

if ispath("./results_lid/$expr_name") == false
    mkdir("./results_lid/$expr_name")
end

if ispath(fig_path) == false
    mkdir(fig_path)
    mkdir(string(fig_path, "/conditions"))
end

if ispath(ckpt_path) == false
    mkdir(ckpt_path)
end

cp("./config2.yaml", config_path; force=true)

"./results_lid/Lid-6s7r1c-03/config.yaml"

In [97]:
function load_exp(filename)
    exp_data = readdlm(filename)  # [t, T, m]
    index = indexin(unique(exp_data[:, 1]), exp_data[:, 1])
    exp_data = exp_data[index, :]
    exp_data[:, 3] = exp_data[:, 3] / maximum(exp_data[:, 3])
    return exp_data
end

load_exp (generic function with 1 method)

In [99]:
l_exp_data = [];
l_exp_info = zeros(Float64, length(l_exp), 3);
for (i_exp, value) in enumerate(l_exp)
    filename = string("data_lid/expdata_no", string(value), ".txt")

    exp_data = Float64.(load_exp(filename))
    
    if value == 1
       exp_data = exp_data[1:90, :]
    elseif value == 2
       exp_data = exp_data[1:90, :]
    elseif value == 3
       exp_data = exp_data[1:90, :]
    elseif value == 4
       exp_data = exp_data[1:90, :]
    elseif value == 5
       exp_data = exp_data[1:90, :]
    elseif value == 6
       exp_data = exp_data[1:90, :]
    elseif value == 7
       exp_data = exp_data[1:90, :]
    elseif value == 8
       exp_data = exp_data[1:90, :]
    elseif value == 9
    #     exp_data = exp_data[1:90, :]
    # elseif value == 16
    #    exp_data = exp_data[1:90, :]
    # elseif value == 17
    #    exp_data = exp_data[1:90, :]
    # elseif value == 18
    #    exp_data = exp_data[1:90, :]
    end

    push!(l_exp_data, exp_data)
    l_exp_info[i_exp, 1] = exp_data[1, 2] # initial temperature, K
end
l_exp_info[:, 2] = readdlm("data_lid/beta.txt")[l_exp];
l_exp_info[:, 3] = (readdlm("data_lid/cata_conc.txt")[l_exp]);

In [100]:
l_exp_info

30×3 Matrix{Float64}:
 350.0    2.5  0.0
 350.0    2.5  0.0
 350.0    2.5  0.0
 350.01   2.5  0.0
 350.01   2.5  0.0
 350.03   2.5  0.0
 350.02   5.0  0.01
 350.03   5.0  0.01
 350.04   5.0  0.01
 350.04   5.0  0.01
 350.04   5.0  0.01
 350.13   5.0  0.01
 350.04   5.0  0.0
   ⋮           
 350.2   10.0  0.01
 350.09  10.0  0.01
 350.21  10.0  0.01
 350.1   10.0  0.01
 350.05  10.0  0.01
 350.22  10.0  0.01
 350.1   10.0  0.0
 350.14  10.0  0.0
 350.11  10.0  0.0
 350.0   10.0  0.0
 350.03  10.0  0.0
 350.01  10.0  0.0

In [101]:
l_exp_data[15]

98×3 Matrix{Float64}:
    0.0  350.04  1.0
   90.0  361.24  0.993207
  180.0  372.36  0.986764
  270.0  383.33  0.979157
  360.0  393.95  0.97485
  450.0  404.15  0.9717
  540.0  413.83  0.969758
  630.0  422.95  0.967861
  720.0  431.64  0.966735
  810.0  439.9   0.96653
  900.0  447.71  0.966815
  990.0  455.23  0.963982
 1080.0  462.55  0.960306
    ⋮            
 4753.0  765.52  0.0422755
 4843.0  772.96  0.0383294
 4933.0  780.5   0.0354607
 5023.0  787.93  0.0335294
 5113.0  795.33  0.0304186
 5203.0  802.78  0.028977
 5293.0  810.21  0.0280804
 5383.0  817.67  0.027098
 5473.0  825.14  0.0272811
 5563.0  832.64  0.0213981
 5653.0  840.13  0.0162015
 5743.0  847.56  0.0132019

In [105]:
Random.seed!(123)
np = nr * (ns + nc + 3) + 1
p = randn(Float64, np) .* 5.e-2;
p[1:nr] .+= 0.8;  # w_lnA
#p[nr*(nss+1)+1:nr*(ns+1)].*= 10           #w_cat_in
p[nr*(ns+1)+1:nr*(ns+nc+1)] .= 0         #w_cat_out
p[nr*(ns+nc+1)+1:nr*(ns+nc+2)] .+= 0.8;  # w_Ea
p[nr*(ns+nc+2)+1:nr*(ns+nc+3)] .+= 0.1;  # w_b
p[end] = 0.1;  # slope

In [107]:
function p2vec(p)
    slope = p[end] .* 1.e1
    w_b = p[1:nr] .* (slope * 10.0)
    w_b = clamp.(w_b, -23, 50)

    w_out = reshape(p[nr+1:nr*(nss+1)], nss, nr)
    @. w_out[1, :] = clamp(w_out[1, :], -3.0, 0.0) #Xylan consume only
    @. w_out[end, :] = clamp(abs(w_out[end, :]), 0.0, 3.0) #Volatile produce only

    if p_cutoff > 0.0
        w_out[findall(abs.(w_out) .< p_cutoff)] .= 0.0
    end

    w_out[nss-1:nss-1, :] .=
        -sum(w_out[1:nss-2, :], dims = 1) .- sum(w_out[nss:nss, :], dims = 1)

    w_cat_in = p[nr*(nss+1)+1:nr*(ns+1)]
    w_cat_out = p[nr*(ns+1)+1:nr*(ns+nc+1)]*0
    
    #w_cat_out = - sign.(w_cat_in) .* abs.(w_cat_out);
    w_cat_in = abs.(w_cat_in)
    
    w_in_Ea = abs.(p[nr*(ns+nc+1)+1:nr*(ns+nc+2)].* (slope * 100.0))
    w_in_Ea = clamp.(w_in_Ea, 0.0, 300.0)

    w_in_b = abs.(p[nr*(ns+nc+2)+1:nr*(ns+nc+3)])

    #if p_cutoff > 0.0
    #    w_in_ocen[findall(abs.(w_in_ocen) .< p_cutoff)] .= 0.0
    #end

    w_in = vcat(clamp.(-w_out, 0.0, 3.0), w_cat_in', w_in_Ea', w_in_b')
    w_out = vcat(w_out, w_cat_out')
    
    return w_in, w_b, w_out
end

p2vec (generic function with 1 method)

In [109]:
pwd()

"C:\\Users\\HP\\Code\\Master thesis\\Model\\Xylan\\Model Diff+Lid"

In [111]:
expr_name1 = "ML-5s5r1c-02"
ckpt_path1 = string("./Good result/", expr_name1, "/checkpoint")
@load string(ckpt_path1, "/mymodel.bson") p 
# ckpt_path2 = string("Code/Master thesis/Model/Xylan/Catalyst model/results_catalyst/Good result/ML-5s5r1c-04/checkpoint") #Code/MODELS/Good model/results_cat3/7s8r1c-8000-02/checkpoint/mymodel.bson
# @load string(ckpt_path2,"/mymodel.bson") p #opt l_loss_train l_loss_val list_grad iter

In [113]:
function display_p(p)
    w_in, w_b, w_out = p2vec(p)
    println("\n species (column) reaction (row)")
    println("w_in | w_cat_in | Ea | b | lnA | w_out | w_cat_out")
    show(stdout, "text/plain", round.(hcat(w_in', w_b, w_out'), digits = 2))
    # println("\n w_out")
    # show(stdout, "text/plain", round.(w_out', digits=3))
    println("\n")
end
display_p(p)


 species (column) reaction (row)
w_in | w_cat_in | Ea | b | lnA | w_out | w_cat_out
5×15 Matrix{Float64}:
 -0.0  2.02  0.18  0.0   0.0  0.12  209.24  0.34  41.94   0.0  -2.02  -0.18   1.99  0.21  0.0
 -0.0  0.0   3.0   0.0   0.0  0.01  140.72  0.01  23.0    0.0   0.6   -3.01   2.4   0.02  0.0
 -0.0  0.15  0.0   0.63  0.0  0.16  189.23  0.2   34.57   0.0  -0.15   0.75  -0.63  0.03  0.0
  3.0  0.0   0.0   0.0   0.0  0.21  224.36  0.57  48.85  -3.0   0.05   0.06   2.44  0.45  0.0
 -0.0  0.0   0.0   1.63  0.0  0.14   62.89  0.01  25.64   0.0   0.5    0.36  -1.63  0.78  0.0



In [114]:
w_in, w_b, w_out = p2vec(p); #getting the trained values

In [115]:
Random.seed!(123)
nr_q = 2
ns_q = 6
nc_q = 1
nss_q = ns_q - nc_q

# p_cutoff = 0.0
q_cutoff = 0
np_q = nr_q * (ns_q + nc_q + 3) + 1 #check the formula
q = randn(Float64, np_q) .* 5.e-2;
q[1:nr_q] .+= 0.8;  # w_lnA
#p[nr*(nss+1)+1:nr*(ns+1)].*= 10           #w_cat_in
q[nr_q*(ns_q+1)+1:nr_q*(ns_q+nc_q+1)] .= 0         #w_cat_outp
q[nr_q*(ns_q+nc_q+1)+1:nr_q*(ns_q+nc_q+2)] .+= 0.8;  # w_Ea
q[nr_q*(ns_q+nc_q+2)+1:nr_q*(ns_q+nc_q+3)] .+= 0.1;  # w_b
q[end] = 0.1;  # slope

In [116]:
function q2vec(q)
    slope_q = q[end] .* 1.e1
    w_b_q = q[1:nr_q] .* (slope_q * 10.0)
    w_b_q = clamp.(w_b_q, 0, 50)

    w_out_q = reshape(q[nr_q+1:nr_q*(nss_q+1)], nss_q, nr_q)
    @. w_out_q[1, :] = clamp(w_out_q[1, :], -3.0, 0.0)
    @. w_out_q[end, :] = clamp(abs(w_out_q[end, :]), -3.0, 0.0)

    if q_cutoff > 0.0
       w_out[findall(abs.(w_out) .< q_cutoff)] .= 0.0
    end

    w_out_q[nss_q-1:nss_q-1, :] .=
        -sum(w_out_q[1:nss_q-2, :], dims = 1) .- sum(w_out_q[nss_q:nss_q, :], dims = 1)

    w_cat_in_q = q[nr_q*(nss_q+1)+1:nr_q*(ns_q+1)]
    w_cat_out_q = q[nr_q*(ns_q+1)+1:nr_q*(ns_q+nc_q+1)]*0
    
    #w_cat_out = - sign.(w_cat_in) .* abs.(w_cat_out);
    w_cat_in_q = abs.(w_cat_in_q)
    
    w_in_Ea_q = abs.(q[nr_q*(ns_q+nc_q+1)+1:nr_q*(ns_q+nc_q+2)].* (slope_q * 100.0))
    w_in_Ea_q = clamp.(w_in_Ea_q, 0.0, 200.0)

    w_in_b_q = abs.(q[nr_q*(ns_q+nc_q+2)+1:nr_q*(ns_q+nc_q+3)])

    w_in_q = vcat(clamp.(-w_out_q, 0.0, 3.0), w_cat_in_q', w_in_Ea_q', w_in_b_q')
    w_out_q = vcat(w_out_q, w_cat_out_q')
    
    w_in_q = hcat(w_in, w_in_q) #changed code
    w_b_q = vcat(w_b, w_b_q) #changed code
    w_out_q = hcat(w_out, w_out_q) #changed code
    
    return w_in_q, w_b_q, w_out_q
end

q2vec (generic function with 1 method)

In [117]:
function display_q(q)
    w_in_q, w_b_q, w_out_q = q2vec(q)
    println("\n species (column) reaction (row)")
    println("w_in | w_cat_in | Ea | b | lnA | w_out | w_cat_out")
    show(stdout, "text/plain", round.(hcat(w_in_q', w_b_q, w_out_q'), digits = 2))
    # println("\n w_out")
    # show(stdout, "text/plain", round.(w_out', digits=3))
    println("\n")
end
display_q(q)


 species (column) reaction (row)
w_in | w_cat_in | Ea | b | lnA | w_out | w_cat_out
7×15 Matrix{Float64}:
 -0.0   2.02  0.18  0.0    0.0  0.12  209.24  0.34  41.94   0.0   -2.02  -0.18   1.99  0.21  0.0
 -0.0   0.0   3.0   0.0    0.0  0.01  140.72  0.01  23.0    0.0    0.6   -3.01   2.4   0.02  0.0
 -0.0   0.15  0.0   0.63   0.0  0.16  189.23  0.2   34.57   0.0   -0.15   0.75  -0.63  0.03  0.0
  3.0   0.0   0.0   0.0    0.0  0.21  224.36  0.57  48.85  -3.0    0.05   0.06   2.44  0.45  0.0
 -0.0   0.0   0.0   1.63   0.0  0.14   62.89  0.01  25.64   0.0    0.5    0.36  -1.63  0.78  0.0
  0.06  0.02  0.0   0.0   -0.0  0.0    74.56  0.11   8.4   -0.06  -0.02   0.01   0.06  0.0   0.0
  0.07  0.0   0.01  0.0   -0.0  0.02   83.52  0.11   7.44  -0.07   0.0   -0.01   0.07  0.0   0.0



In [118]:
function getsampletemp(t, T0, beta)
    if beta < 100
        T = T0 + beta / 60 * t  # K/min to K/s
    else
        tc = [999.0, 1059.0] .* 60.0
        Tc = [beta, 370.0, 500.0] .+ 273.0
        HR = 40.0 / 60.0
        if t <= tc[1]
            T = Tc[1]
        elseif t <= tc[2]
            T = minimum([Tc[1] + HR * (t - tc[1]), Tc[2]])
        else
            T = minimum([Tc[2] + HR * (t - tc[2]), Tc[3]])
        end
    end
    return T
end

getsampletemp (generic function with 1 method)

In [123]:
const R = -1.0 / 8.314e-3  # universal gas constant, kJ/mol*K
@inbounds function crnn!(du, u, q, t)
    logX = @. log(clamp(u, lb, 10.0))
    T = getsampletemp(t, T0, beta)
    w_in_x_q = w_in_q' * vcat(logX, R / T, log(T))
    du .= w_out_q * (@. exp(w_in_x_q + w_b_q))
end

crnn! (generic function with 1 method)

In [124]:
tspan = [0.0, 1.0];
u0 = zeros(ns);
u0[1] = 1.0;
prob = ODEProblem(crnn!, u0, tspan, q , abstol = lb)

ODEProblem with uType Vector{Float64} and tType Float64. In-place: true
timespan: (0.0, 1.0)
u0: 6-element Vector{Float64}:
 1.0
 0.0
 0.0
 0.0
 0.0
 0.0

In [125]:
condition(u, t, integrator) = u[1] < lb * 10.0
affect!(integrator) = terminate!(integrator)
_cb = DiscreteCallback(condition, affect!)

alg = TRBDF2();
#alg = RK4();
#alg = TRBDF2();
#alg = AutoTsit5(Rosenbrock23(autodiff=false));
#alg = Rosenbrock23(autodiff=false);
#alg = AutoVern7(Rodas5P(autodiff = false));
#sense = BacksolveAdjoint(checkpointing=true; autojacvec=ZygoteVJP());
sense = ForwardSensitivity(autojacvec = true)

ForwardSensitivity{0, true, Val{:central}}(true, false)

In [126]:
function pred_n_ode(q, i_exp, exp_data)
    #global T0, beta, ocen = l_exp_info[i_exp, :]
    global T0, beta, cat_conc = l_exp_info[i_exp, :]
    #global w_in, w_b, w_out = p2vec(p)
    global w_in_q, w_b_q, w_out_q = q2vec(q)
    
    # rows_to_keep_in = setdiff(1:size(w_in, 1), [2,5]) 
    # rows_to_keep_out = setdiff(1:size(w_out, 1), [2,5])
    # w_in = w_in[rows_to_keep_in, :]
    # w_out = w_out[rows_to_keep_out, :]

    ts = @view(exp_data[:, 1])
    tspan = [ts[1], ts[end]]
    u0[end] = cat_conc 
    sol = solve(
        prob,
        alg,
        tspan = tspan,
        p = q,
        saveat = ts, 
        sensealg = sense,
        maxiters = maxiters
        # callback = _cb,
    )

    if sol.retcode == :Success
        nothing
    else
        #@sprintf("solver failed beta: %.0f ocen: %.2f", beta, exp(ocen))
        @sprintf("solver failed beta: %.0f", beta)
    end
    if length(sol.t) > length(ts)
        # @show exp_data[:, 1]
        # @show sol.t
        return  sol[:, 1:length(ts)]
    else
        return sol
    end
end

pred_n_ode (generic function with 1 method)

In [127]:
function loss_neuralode(q, i_exp)
    exp_data = l_exp_data[i_exp]
    pred = clamp.(Array(pred_n_ode(q, i_exp, exp_data)), 0 , Inf)
    masslist = sum(clamp.(@view(pred[1:end-1-nc, :]), 0, Inf), dims = 1)'
    gaslist = clamp.(@views(pred[end-nc, :]), 0, Inf)

    loss = mae(masslist, @view(exp_data[1:length(masslist), 3])) + mae(gaslist, 1 .- @view(exp_data[1:length(masslist), 3]))
    return loss
end

loss_neuralode (generic function with 1 method)

In [128]:
@time loss = loss_neuralode(q, 1)

 19.018881 seconds (30.68 M allocations: 1.706 GiB, 7.49% gc time, 99.75% compilation time: 56% of which was recompilation)


0.12624768238105263

In [129]:
# --- Early stopping parameters ---
# best_val_loss      = Inf
# best_epoch        = 0
# epochs_no_improve = 0
# patience          = 2500      # you can adjust this
# best_p            = copy(p)  # store the best weights

# === Trace storage for plotting === do not run when visualising old results
# lr_trace           = Float64[]
# train_loss_trace   = Float64[]
# val_loss_trace     = Float64[]
# grad_trace         = Float64[]

In [130]:
function plot_sol(i_exp, sol, exp_data, Tlist, cap, sol0 = nothing)
    T0, beta, cat = l_exp_info[i_exp, :]
    #T0, beta = l_exp_info[i_exp, :]
    ts = sol.t / 60.0
    Tlist = [getsampletemp(t, T0, beta) for t in sol.t]
    ind = length(ts)
    plt = plot(
        exp_data[:, 2],
        exp_data[:, 3],
        seriestype = :scatter,
        label = "Exp",
    )

    plot!(
        plt,
        Tlist,
        sum(clamp.(sol[1:end-1-nc, :], 0, Inf), dims = 1)',
        lw = 3,
        legend = :left,
        label = "CRNN",
    )
    
    if sol0 !== nothing
        plot!(
            plt,
            Tlist0 = [getsampletemp(t, T0, beta) for t in sol0.t],
            sum(sol0[1:end-1-nc, :], dims = 1)',
            label = "initial model",
        )
    end
    xlabel!(plt, "Temperature [K]")
    ylabel!(plt, "Normalized Mass")
    ylims!(0,1.1)
    title!(plt, cap)
    exp_cond = string(
        @sprintf(
            #"T0 = %.1f K \n beta = %.2f K/min",
            "T0 = %.1f K \n beta = %.2f K/min \n [catalyst] = %.2f",
            T0,
            beta,
            cat*100
        )
    )
    annotate!(plt, Tlist[end] * 0.85, 0.4, exp_cond)

    # plt2 = twinx()
    # plot!(
    #     plt2,
    #     exp_data[1:ind, 1] / 60,
    #     Tlist,
    #     lw = 2,
    #     ls = :dash,
    #     legend = :topright,
    #     label = "T",
    # )
    # ylabel!(plt2, "Temp")

#     p2 = plot(ts, sol[1, :], lw = 2, legend = :right, label = "Cellulose")
#     for i = 2:ns-nc
#         plot!(p2, ts, sol[i, :], lw = 2, label = "S$i")
#     end
#     xlabel!(p2, "Time [min]")
#     ylabel!(p2, "Mass")

#     plt = plot(plt, p2, framestyle = :box, layout = @layout [a; b])
#     plot!(plt, size = (800, 800))
#     return plt
# end
    
    p2 = plot(Tlist, sol[1, :], lw = 4, legend = :right, label = "Xylan")
    for i = 2:nss-1
        plot!(p2, Tlist, sol[i, :], lw = 4, label = "S$i")
    end
    plot!(p2, Tlist, sol[ns-nc, :], lw = 4, label = "Volatile")
    xlabel!(p2, "Temperature [K]")
    ylabel!(p2, "Mass")

    p3 = plot(ts, sol[1, :], lw = 4, legend = :right, label = "Xylan")
    for i = 2:nss-1
        plot!(p3, ts, sol[i, :], lw = 4, label = "S$i")
    end
    plot!(p3, ts, sol[ns-nc, :], lw = 4, label = "Volatile")
    xlabel!(p3, "Time [min]")
    ylabel!(p3, "Mass")

    plt = plot(plt, p2, p3, framestyle = :box, layout = @layout [a; b; c])
    plot!(plt, size = (800, 800))
    return plt
end

cbi = function (q, i_exp)
    exp_data = l_exp_data[i_exp]
    sol = pred_n_ode(q, i_exp, exp_data)
    Tlist = similar(sol.t)
    #T0, beta, ocen = l_exp_info[i_exp, :]
    T0, beta = l_exp_info[i_exp, :]
    for (i, t) in enumerate(sol.t)
        Tlist[i] = getsampletemp(t, T0, beta)
    end
    value = l_exp[i_exp]
    plt = plot_sol(i_exp, sol, exp_data, Tlist, "exp_$value")
    png(plt, string(fig_path, "/conditions/pred_exp_$value"))
    return false
end


function plot_loss(l_loss_train, l_loss_val; yscale = :log10)
    plt_loss = plot(l_loss_train, yscale = yscale, label = "train")
    plot!(plt_loss, l_loss_val, yscale = yscale, label = "val")
    plt_grad = plot(list_grad, yscale = yscale, label = "grad_norm")
    xlabel!(plt_loss, "Epoch")
    ylabel!(plt_loss, "Loss")
    xlabel!(plt_grad, "Epoch")
    ylabel!(plt_grad, "Gradient Norm")
    # ylims!(plt_loss, (-Inf, 1e0))
    # ylims!(plt_grad, (-Inf, 1e3))
    # plt_all = plot([plt_loss, plt_grad]..., legend = :top, framestyle=:box)
    # plot!(
    #     plt_all,
    #     size=(900, 400),
    #     xtickfontsize = 11,
    #     ytickfontsize = 11,
    #     xguidefontsize = 12,
    #     yguidefontsize = 12,
    # )
    # png(plt_all, string(fig_path, "/loss_grad"))
    png(plt_loss, string(fig_path, "/loss_plot"))
    png(plt_grad, string(fig_path, "/grad_plot"))
end

l_loss_train = []
l_loss_val = []
list_grad = []
iter = 1
# cb = function (q, loss_train, loss_val, g_norm)
cb = function (q, loss_train, loss_val, g_norm, mlf, exprun)
    global l_loss_train, l_loss_val, list_grad, iter
    push!(l_loss_train, loss_train)
    push!(l_loss_val, loss_val)
    push!(list_grad, g_norm)

    println("LOGGING metrics: iter = ", iter,
        " train = ", loss_train,
        " val = ", loss_val,
        " grad = ", g_norm)
    # Log metrics to MLFlow
    logmetric(mlf, exprun, "train_loss", loss_train, step=iter)
    logmetric(mlf, exprun, "val_loss", loss_val, step=iter)
    logmetric(mlf, exprun, "grad_norm", g_norm, step=iter)
    
    #if iter % n_plot == 0 #the below code saves the code with the lowest loss validation
    if all(x -> loss_val < x, l_loss_val[1:end-1])
        display_q(q)
        list_exp = randperm(n_exp)[1]
        @printf(
            "Min Loss train: %.2e val: %.2e",
            minimum(l_loss_train),
            minimum(l_loss_val)
        )
        println("\n update plot ", l_exp[list_exp], "\n")
        for i_exp in list_exp
            cbi(q, i_exp)
        end

        plot_loss(l_loss_train, l_loss_val; yscale = :log10)

        @save string(ckpt_path, "/mymodel.bson") q opt l_loss_train l_loss_val list_grad iter #lr_trace train_loss_trace val_loss_trace grad_trace
    end
    iter += 1 
end

if is_restart
    @load string(ckpt_path, "/mymodel.bson") q opt l_loss_train l_loss_val list_grad iter #lr_trace train_loss_trace val_loss_trace grad_trace
    iter += 1
end

# # Diagnostic function to check your current setup
# function diagnose_species_indexing()
#     println("Current setup:")
#     println("  ns (total species): $ns")
#     println("  nr (reactions): $nr") 
#     println("  nc (catalysts): $nc")
#     println("  nss (non-catalyst species): $nss")
#     println("\nSpecies indexing:")
#     println("  Xylan: 1")
#     for i in 2:nss-1
#         println("  Intermediate S$(i-1): $i")
#     end
#     if nc > 0
#         println("  Volatile: $(nss)")
#         println("  Catalyst: $(ns)")
#     else
#         println("  Volatile: $(ns)")
#     end
    
#     println("\nFor mass calculation (solid species only):")
#     if nc > 0
#         println("  Include indices: 1:$(ns-nc) (excludes catalyst)")
#     else
#         println("  Include indices: 1:$(ns-1) (excludes volatile)")
#     end
# end

# # Run this to check your setup
# diagnose_species_indexing()

In [131]:
# Log hyperparameters to MLFlow (once, before training loop)
logparam(mlf, exprun, "ns", string(ns))
logparam(mlf, exprun, "nr", string(nr))
logparam(mlf, exprun, "nc", string(nc))

logparam(mlf, exprun, "lb", string(lb))
logparam(mlf, exprun, "n_epoch", string(n_epoch))
logparam(mlf, exprun, "n_plot", string(n_plot))
logparam(mlf, exprun, "grad_max", string(grad_max))
logparam(mlf, exprun, "maxiters_1", string(maxiters))

logparam(mlf, exprun, "lr_max", string(lr_max))
logparam(mlf, exprun, "lr_min", string(lr_min))
logparam(mlf, exprun, "lr_decay", string(lr_decay))
logparam(mlf, exprun, "lr_decay_step", string(lr_decay_step))
logparam(mlf, exprun, "w_decay", string(w_decay))

# Log that this run was extended
# logparam(mlf, exprun, "extended_run", "true")

# Log the new hyperparameters used in the extension
# logparam(mlf, exprun, "lr_max_new", string(lr_max))
# logparam(mlf, exprun, "lr_decay_new", string(lr_decay))
# logparam(mlf, exprun, "lr_decay_step_new", string(lr_decay_step))
# logparam(mlf, exprun, "n_epoch_extended_1", string(n_epoch))
# logparam(mlf, exprun, "w_decay_extended", string(w_decay))

true

In [143]:
epochs = ProgressBar(iter:n_epoch);
loss_epoch = zeros(Float64, n_exp);
grad_norm = zeros(Float64, n_exp);
for epoch in epochs
    global q
    for i_exp in randperm(n_exp)
        if i_exp in l_val
            continue
        end
        grad = ForwardDiff.gradient(x -> loss_neuralode(x, i_exp), q)
        grad_norm[i_exp] = norm(grad, 2)
        if grad_norm[i_exp] > grad_max
            grad = grad ./ grad_norm[i_exp] .* grad_max
        end
        update!(opt, q, grad)
    end
    for i_exp = 1:n_exp
        loss_epoch[i_exp] = loss_neuralode(q, i_exp)
    end
    loss_train = mean(loss_epoch[l_train])
    loss_val = mean(loss_epoch[l_val])
    grad_mean = mean(grad_norm[l_train])
    # === Log learning rate and losses ===
    # push!(lr_trace, opt[1].eta)
    # push!(train_loss_trace, loss_train)
    # push!(val_loss_trace, loss_val)
    # push!(grad_trace, grad_mean)
    set_description(
        epochs,
        string(
            @sprintf(
                "Loss train: %.2e val: %.2e grad: %.2e lr: %.1e",
                loss_train,
                loss_val,
                grad_mean,
                opt[1].eta
                #opt.eta
            )
        ),
    )
    # cb(q, loss_train, loss_val, grad_mean)
    cb(q, loss_train, loss_val, grad_mean, mlf, exprun)

    # --- Early stopping logic ---
    # if loss_val < best_val_loss
    #     best_val_loss = loss_val
    #     best_epoch    = epoch
    #     best_p        = copy(p)
    #     epochs_no_improve = 0
    # else
    #     epochs_no_improve += 1
    # end
    # if epochs_no_improve >= patience
    #     println("Early stopping: best val loss = $(best_val_loss) at epoch $(best_epoch)")
    #     break
    # end
end

conf["loss_train"] = minimum(l_loss_train)
conf["loss_val"] = minimum(l_loss_val)
YAML.write_file(config_path, conf)

0.0%┣                                             ┫ 0/1.5k [00:00<00:-8, -0s/it]


LOGGING metrics: iter = 1 train = 0.19775023924576163 val = 0.21473412127694566 grad = 2.8172160218541533

 species (column) reaction (row)
w_in | w_cat_in | Ea | b | lnA | w_out | w_cat_out
7×15 Matrix{Float64}:
 -0.0   2.02  0.18  0.0    0.0  0.12  209.24  0.34  41.94   0.0   -2.02  -0.18   1.99  0.21  0.0
 -0.0   0.0   3.0   0.0    0.0  0.01  140.72  0.01  23.0    0.0    0.6   -3.01   2.4   0.02  0.0
 -0.0   0.15  0.0   0.63   0.0  0.16  189.23  0.2   34.57   0.0   -0.15   0.75  -0.63  0.03  0.0
  3.0   0.0   0.0   0.0    0.0  0.21  224.36  0.57  48.85  -3.0    0.05   0.06   2.44  0.45  0.0
 -0.0   0.0   0.0   1.63   0.0  0.14   62.89  0.01  25.64   0.0    0.5    0.36  -1.63  0.78  0.0
  0.05  0.02  0.0   0.0   -0.0  0.0    76.01  0.11   8.53  -0.05  -0.02   0.01   0.06  0.0   0.0
  0.07  0.0   0.01  0.0   -0.0  0.02   85.12  0.11   7.55  -0.07   0.0   -0.01   0.07  0.0   0.0

Min Loss train: 1.98e-01 val: 2.15e-01
 update plot 7



Loss train: 1.98e-01 val: 2.15e-01 grad: 2.82e+00 lr: 1.0e-04 0.1%┣┫ 1/1.5k [01:29<Inf:Inf, InfGs/it]


LOGGING metrics: iter = 2 train = 0.19047830859209547 val = 0.19161469529588285 grad = 2.3496376948449504

 species (column) reaction (row)
w_in | w_cat_in | Ea | b | lnA | w_out | w_cat_out
7×15 Matrix{Float64}:
 -0.0   2.02  0.18  0.0    0.0  0.12  209.24  0.34  41.94   0.0   -2.02  -0.18   1.99  0.21  0.0
 -0.0   0.0   3.0   0.0    0.0  0.01  140.72  0.01  23.0    0.0    0.6   -3.01   2.4   0.02  0.0
 -0.0   0.15  0.0   0.63   0.0  0.16  189.23  0.2   34.57   0.0   -0.15   0.75  -0.63  0.03  0.0
  3.0   0.0   0.0   0.0    0.0  0.21  224.36  0.57  48.85  -3.0    0.05   0.06   2.44  0.45  0.0
 -0.0   0.0   0.0   1.63   0.0  0.14   62.89  0.01  25.64   0.0    0.5    0.36  -1.63  0.78  0.0
  0.05  0.02  0.0   0.0   -0.0  0.0    77.43  0.1    8.65  -0.05  -0.02   0.02   0.06  0.0   0.0
  0.07  0.0   0.01  0.0   -0.0  0.02   86.68  0.1    7.66  -0.07   0.0   -0.01   0.08  0.0   0.0

Min Loss train: 1.90e-01 val: 1.92e-01
 update plot 9



Loss train: 1.90e-01 val: 1.92e-01 grad: 2.35e+00 lr: 1.0e-04 0.1%┣┫ 2/1.5k [01:31<37:50:02, 91s/it]


...

Loss train: 1.53e-01 val: 7.40e-02 grad: 6.51e-01 lr: 2.5e-05 99.9%┣┫ 1.5k/1.5k [50:59<00:04, 2s/it]


LOGGING metrics: iter = 1499 train = 0.152865980117599 val = 0.07396320122169862 grad = 0.6691750805697579


Loss train: 1.53e-01 val: 7.40e-02 grad: 6.69e-01 lr: 2.5e-05 99.9%┣┫ 1.5k/1.5k [51:02<00:02, 2s/it]


LOGGING metrics: iter = 1500 train = 0.15286830125209383 val = 0.07377090271206195 grad = 0.6518178873470771


Loss train: 1.53e-01 val: 7.38e-02 grad: 6.52e-01 lr: 1.3e-05 100.0%┣┫ 1.5k/1.5k [51:04<00:00, 2s/it]
Loss train: 1.53e-01 val: 7.38e-02 grad: 6.52e-01 lr: 1.3e-05 100.0%┣┫ 1.5k/1.5k [51:04<00:00, 2s/it]


In [ ]:
# length(lr_trace)
# iter

In [ ]:
# # After training, optionally restore the best weights:
# p = best_p
# println("✅ Best validation loss: $best_val_loss at epoch $best_epoch")

# println("lengths: ",
#         length(lr_trace), " ",
#         length(train_loss_trace), " ",
#         length(val_loss_trace), " ",
#         length(grad_trace))


In [175]:
# epochs = 1:length(lr_trace)

# plot(epochs, lr_trace, yscale=:log10, xlabel="Epoch", ylabel="Learning Rate", label="LR", lw=2, legend=:topright)
# plot!(epochs, train_loss_trace, label="Train Loss", lw=2)
# plot!(epochs, val_loss_trace, label="Val Loss", lw=2)
# # savefig(p, "lr_loss_curves.png")
# # plot(epochs, grad_trace, xlabel="Epoch", ylabel="Gradient Norm", label="Grad", lw=2)

LoadError: UndefVarError: `lr_trace` not defined

In [81]:
expr_name = "Lid-6s6r1c-02"
fig_path = string("./results_lid/", expr_name, "/figs")
ckpt_path = string("./results_lid/", expr_name, "/checkpoint")
@load string(ckpt_path, "/mymodel.bson") q opt l_loss_train l_loss_val list_grad iter

p_cutoff = 0.0

display_q(q)

for i_exp = 1:n_exp
    cbi(q, i_exp)
end

loss_epoch = zeros(Float64, n_exp);
for i_exp = 1:n_exp
    loss_epoch[i_exp] = loss_neuralode(q, i_exp)
end
loss_train = mean(loss_epoch[l_train])
loss_val = mean(loss_epoch[l_val])
loss_train, loss_val


 species (column) reaction (row)
w_in | w_cat_in | Ea | b | lnA | w_out | w_cat_out
6×15 Matrix{Float64}:
 -0.0  2.02  0.18  0.0    0.0  0.12  209.24  0.34  41.94   0.0  -2.02  -0.18   1.99  0.21  0.0
 -0.0  0.0   3.0   0.0    0.0  0.01  140.72  0.01  23.0    0.0   0.6   -3.01   2.4   0.02  0.0
 -0.0  0.15  0.0   0.63   0.0  0.16  189.23  0.2   34.57   0.0  -0.15   0.75  -0.63  0.03  0.0
  3.0  0.0   0.0   0.0    0.0  0.21  224.36  0.57  48.85  -3.0   0.05   0.06   2.44  0.45  0.0
 -0.0  0.0   0.0   1.63   0.0  0.14   62.89  0.01  25.64   0.0   0.5    0.36  -1.63  0.78  0.0
 -0.0  0.07  0.0   0.05  -0.0  0.07  123.31  0.09  12.3    0.0  -0.07   0.12  -0.05  0.0   0.0



(0.15639660319895407, 0.0642247797092126)

In [ ]:
# pyplot()
plot_loss(l_loss_train, l_loss_val; yscale = :log10)

In [ ]:
function loss_neuralode_res(p)
    l_loss_exp = zeros(n_exp)
    for i_exp in 1:n_exp
        exp_data = l_exp_data[i_exp]
        pred = Array(pred_n_ode(p, i_exp, exp_data))
        masslist = sum(clamp.(@view(pred[1:end-1, :]), 0, Inf), dims = 1)'
        l_loss_exp[i_exp] = mae(masslist, @view(exp_data[1:length(masslist), 3]))
    end
    return l_loss_exp
end

In [ ]:
println("results after pruning")
maxiters = 1e5
p_cutoff = 0.0

In [ ]:
loss_epoch = zeros(Float64, n_exp);
for i_exp = 1:n_exp
    loss_epoch[i_exp] = loss_neuralode(p, i_exp)
end
loss_train = mean(loss_epoch[l_train])
loss_val = mean(loss_epoch[l_val])
@printf(
    "Loss train: %.2f val: %.2f p_cutoff: %.2e",
    log10(loss_train),
    log10(loss_val),
    p_cutoff
)

display_p(p)

In [ ]:
for i_exp in randperm(n_exp)
    cbi(p, i_exp)
end

In [ ]:
l_loss_exp = loss_neuralode_res(p)

In [ ]:
l_exp_data = []
l_exp_info = zeros(Float64, length(l_exp), 3)
for (i_exp, value) in enumerate(l_exp)
    filename = string("lid_data/expdata_no", string(value), ".txt")
    exp_data = Float64.(load_exp(filename))
    push!(l_exp_data, exp_data)
    l_exp_info[i_exp, 1] = exp_data[1, 2] # initial temperature, K
end
l_exp_info[:, 2] = readdlm("lid_data/beta.txt")[l_exp]
#l_exp_info[:, 3] = log.(readdlm("exp_data_cat3/ocen.txt")[l_exp] .+ llb)
l_exp_info[:, 3] = (readdlm("lid_data/cata_conc.txt")[l_exp]);

l_loss_exp = loss_neuralode_res(p)

conc_end = zeros(n_exp, ns)

In [ ]:
# Plot TGA data
list_plt = []
for i_exp = 1:18
    T0, beta, cata_conc = l_exp_info[i_exp, :]
    exp_data = l_exp_data[i_exp]
    sol = pred_n_ode(p, i_exp, exp_data)
    Tlist = similar(sol.t)
    T0, beta, cata_conc = l_exp_info[i_exp, :]
    for (i, t) in enumerate(sol.t)
        Tlist[i] = getsampletemp(t, T0, beta)
    end

    conc_end[i_exp, :] .= sol[:, end]

    plt = plot(
        Tlist,
        exp_data[:, 3],
        seriestype = :scatter,
        label = "Exp-$i_exp",
        framestyle = :box,
    )
    plot!(
        plt,
        Tlist,
        sum(sol[1:end-1, :], dims = 1)',
        lw = 2,
        legend = false,
    )
    xlabel!(plt, "Temperature [K]")

    exp_cond = @sprintf("T0:%.0f K", T0)
    exp_cond *= @sprintf("\nbeta:%.0f K/min", beta)

    ann_loc = [0.2, 0.3]
    if i_exp in [6, 7, 14]
        ann_loc = [0.7, 0.4]
    end
    annotate!(plt, xlims(plt)[1] + (xlims(plt)[end] - xlims(plt)[1]) * ann_loc[1],
                   ylims(plt)[1] + (ylims(plt)[end] - ylims(plt)[1]) * ann_loc[2],
                   text(exp_cond, 11))

    ylabel!(plt, "Mass [-] (No. $i_exp)")
    ylims!(plt, (0.0, 1.0))
    plot!(
        plt,
        xtickfontsize = 11,
        ytickfontsize = 11,
        xguidefontsize = 12,
        yguidefontsize = 12,
    )

    push!(list_plt, plt)
end

In [ ]:
plt_all = plot(list_plt..., layout = (7, 2))
plot!(plt_all, size = (800, 1200))
png(plt_all, string(fig_path, "/TGA_mass_summary"))

In [ ]:
# https://github.com/DENG-MIT/Biomass.jl/blob/main/backup/crnn_cellulose_ocen_test.jl
varnames = ["Cellu", "S2", "S3", "Vola"]
for i_exp in 1:n_exp
    T0, beta, ocen = l_exp_info[i_exp, :]
    exp_data = l_exp_data[i_exp]
    sol = pred_n_ode(p, i_exp, exp_data)
    Tlist = similar(sol.t)
    T0, beta, ocen = l_exp_info[i_exp, :]
    for (i, t) in enumerate(sol.t)
        Tlist[i] = getsampletemp(t, T0, beta)
    end
    value = l_exp[i_exp]
    list_plt = []
    scale_factor = 1 ./ maximum(sol[:, :], dims=2)
    scale_factor .= 1.0
    plt = plot(Tlist, clamp.(sol[1, :], 0, Inf), label = varnames[1])
    for i in 2:ns
        if scale_factor[i] > 1.1
            _label =  @sprintf("%s x %.2e", varnames[i], scale_factor[i])
        else
            _label = varnames[i]
        end
        plot!(plt, Tlist, clamp.(sol[i, :], 0, Inf) * scale_factor[i], label = _label)
    end
    xlabel!(plt, "Temperature [K]");
    ylabel!(plt, "Mass (-)");
    plot!(plt, size=(350, 350), legend=:topleft, framestyle=:box)
    plot!(
        plt,
        xtickfontsize = 11,
        ytickfontsize = 11,
        xguidefontsize = 12,
        yguidefontsize = 12,
    )
    png(plt, string(fig_path, "/pred_S_exp_$value"))
end

include("dataset.jl")
gr()